# fit4 — nf4 multiplicative three-loop-PT, order 4

Updated upstream analysis (`2af131f`), Wilson flow, on-the-fly TLN, three-volume infinite-volume limit, and exhaustive integer flow-time-window scan.  The interpolation ansatz is
$$
\beta(x)=\beta_{\rm PT}^{(3)}(x)\left[1+\sum_{n=1}^{4}c_n(x/4\pi)^n\right].
$$

Every plot family has its own cell. Outputs are organized as `fit4/order_4/t_<min>_<max>/<mode>/`, where mode is `diagonal` or `correlated`. The correlated result uses the upstream kernel covariance. Rectangle measurements are combined at the raw-history level into the Symanzik observable before its dedicated TLN correction.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

from pathlib import Path
import copy, gc, json, sys

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'pyproject.toml').is_file():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / 'pyproject.toml').is_file():
    raise RuntimeError('Run this notebook from inside the ContinuousBetaFunction repository')
sys.path.insert(0, str(REPO_ROOT / 'src'))

import betafn
import gvar as gv
import lsqfit
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from IPython.display import display

print('betafn package:', Path(betafn.__file__).resolve())
print('repository:', REPO_ROOT)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'text.usetex': True, 'font.family': 'serif',
    'font.serif': ['Computer Modern'], 'font.size': 15,
    'figure.dpi': 300, 'savefig.dpi': 300,
    'axes.labelsize': 20, 'axes.titlesize': 18,
    'legend.fontsize': 11, 'xtick.labelsize': 18, 'ytick.labelsize': 18,
    'xtick.minor.visible': True, 'ytick.minor.visible': True,
    'grid.alpha': .35, 'grid.linestyle': '-', 'axes.axisbelow': True,
})

## Configuration and complete window catalogue

Change physics/statistical choices only here. Every downstream cell reads these values.

In [ ]:
FIT_ID = 'fit4'
ORDER = 4
FIT_WATERMARK = rf'{FIT_ID}, correction order {ORDER}'
CORRECTION = 'tln'
DATA_DIR = Path('/Users/yaman/Contbetafn/data/New Four')
OUTPUT_ROOT = REPO_ROOT / 'src/betafn' / FIT_ID / f'order_{ORDER}'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

COUPLINGS = ('20p0', '18p0', '16p0', '14p0', '12p0', '10p0', '9p00', '8p50')
VOLUMES = {
    coupling: ('l32l32l32t64', 'l40l40l40t80', 'l48l48l48t96')
    for coupling in COUPLINGS
}
FLOW = 'wilson'
OBSERVABLES = ('p', 'c', 's')
OP_LABELS = {'p': 'Wilson', 'c': 'clover', 's': 'Symanzik'}
# Match the operator convention used in the reference scan notebook.
OP_COLORS = {
    'p': plt.cm.YlOrRd(0.75),
    'c': plt.cm.YlGn(0.75),
    's': plt.cm.Greys(0.75),
}

# Exhaustive integer windows used by the scan reference notebook.
TMIN_VALUES = tuple(range(4, 8))
TMAX_LIMIT = 8
WINDOWS = tuple(
    (float(tmin), float(tmax))
    for tmin in TMIN_VALUES
    for tmax in range(tmin + 1, TMAX_LIMIT + 1)
)
CENTRAL_WINDOW = (4.0, 6.0)
G2_GRID = (0.9, 4.9, 0.2)
TARGET_G2 = (1.1, 1.3, 1.5, 1.8, 2.2, 2.6, 3.0, 4.0)

def window_tag(window):
    return f't_{window[0]:g}_{window[1]:g}'.replace('.', 'p')

def case_dir(window, mode=None):
    path = OUTPUT_ROOT / window_tag(window)
    if mode is not None:
        path = path / mode
    path.mkdir(parents=True, exist_ok=True)
    return path

def save_figure(fig, window, name, mode=None):
    base = case_dir(window, mode) / name
    fig.savefig(base.with_suffix('.png'), dpi=300, bbox_inches='tight')
    fig.savefig(base.with_suffix('.pdf'), dpi=300, bbox_inches='tight')
    plt.show()
    plt.close(fig)

def beta_value(coupling):
    return float(coupling.replace('p', '.'))

def volume_value(volume):
    return np.prod([float(x) for x in volume.replace('t', 'l').split('l')[1:] if x])

def flow_times(window):
    return [t for t in sorted(bf.ntrp_fits[FLOW][OBSERVABLES[0]], key=float)
            if window[0] <= float(t) <= window[1]]

def pt_over_g4(x, loops):
    x = np.asarray(x, dtype=float)
    pt = bf.perturbative_beta_function
    return sum(-coefficient * x**power / pt.nrm**(power + 1)
               for power, coefficient in enumerate(pt.b[:loops]))

print(f'{len(WINDOWS)} windows:', WINDOWS)
print('output root:', OUTPUT_ROOT)

## Analysis construction

The pulled upstream package computes TLN on demand. No external `.tln` path is supplied. The raw `Es` history is the rectangle measurement and is replaced by `(5/3)Ep-(2/3)Es` before averaging and TLN.

In [ ]:
if not DATA_DIR.is_dir():
    raise FileNotFoundError(DATA_DIR)

bf = betafn.BetaFunction(nf=4)
interpolation = bf.perturbative_interpolation(
    loops=3, correction_order=ORDER, free_intercept=False,
    width=10.0, xerrors=True,
)
config = betafn.AnalysisConfig(
    data_path=str(DATA_DIR),
    interpolation=interpolation,
    continuum_window=CENTRAL_WINDOW,
    g2_grid=G2_GRID,
    flows=(FLOW,),
    observables=OBSERVABLES,
    combine={'s': {'p': 5.0 / 3.0, 's': -2.0 / 3.0}},
    couplings=COUPLINGS,
    volumes=VOLUMES,
    correction=CORRECTION,
    binsize=15,
    use_gamma_method=True,
    process_window=(1.5, 10.5),
    fit_window=(2.0, 10.0),
    error_mode='fit',
    cov_mode='kernel',
    correlated=True,
    verbosity=0,
)
display(pd.DataFrame([config.describe()]).T.rename(columns={0: 'fit4 setting'}))

## Stage 1 — processing with TLN

In [ ]:
bf.run_processing(config)
print('correction:', bf._process_config.correction)
print('processed couplings:', sorted(bf.avg_data, key=beta_value))
print('binsize:', bf._binsize)
print(bf.data_report())

## Plot 1 — processed largest-volume data

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for operator in OBSERVABLES:
    first = True
    for coupling in sorted(bf.avg_data, key=beta_value):
        volume = max(bf.avg_data[coupling], key=volume_value)
        _, g2 = bf.processed_series(coupling, volume, '0p00', FLOW, operator, 'g2')
        _, beta = bf.processed_series(coupling, volume, '0p00', FLOW, operator, 'beta')
        ratio = np.asarray([b / x**2 for x, b in zip(g2, beta)], dtype=object)
        ax.errorbar(gv.mean(g2), gv.mean(ratio), yerr=gv.sdev(ratio), fmt='o', ms=2.5,
                    alpha=.6, color=OP_COLORS[operator],
                    label=OP_LABELS[operator] if first else None)
        first = False
ax.set(xlabel=r'$g^2_{GF}$', ylabel=r'$\beta_{GF}/g_{GF}^4$', title='Processed largest-volume data (TLN)')
ax.legend(frameon=False)
base = OUTPUT_ROOT / 'processed_largest_volume_fit4'
fig.savefig(base.with_suffix('.png'), dpi=300, bbox_inches='tight')
fig.savefig(base.with_suffix('.pdf'), dpi=300, bbox_inches='tight')
plt.show(); plt.close(fig)

## Stages 2–4 — chiral, infinite volume, and fixed fit4 interpolation

In [ ]:
bf.run_chiral(config)
bf.run_infinite_volume(config)
bf.run_interpolation(config)
print(bf.stage_summary('chiral'))
print(bf.stage_summary('infinite_volume'))
print(bf.stage_summary('interpolation'))

## Plot 2 — infinite-volume extrapolations for every bare coupling

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
for ax, coupling in zip(axes.ravel(), COUPLINGS):
    times = sorted(bf.iv_fits[coupling]['g2'][FLOW][OBSERVABLES[0]], key=float)
    time = min(times, key=lambda t: abs(float(t) - CENTRAL_WINDOW[0]))
    for operator in OBSERVABLES:
        xfit, yfit, data = bf.infinite_volume_curve(coupling, FLOW, operator, time, x='g2')
        intercept = bf.infinite_volume.model.evaluate(0.0, bf.iv_fits[coupling]['g2'][FLOW][operator][time])
        color = OP_COLORS[operator]
        ax.errorbar(gv.mean(data.x), gv.mean(data.y), xerr=gv.sdev(data.x), yerr=gv.sdev(data.y),
                    fmt='o', capsize=3, color=color, label=OP_LABELS[operator])
        ax.plot(xfit, gv.mean(yfit), color=color)
        ax.fill_between(xfit, gv.mean(yfit)-gv.sdev(yfit), gv.mean(yfit)+gv.sdev(yfit), color=color, alpha=.15)
        ax.errorbar([0], [gv.mean(intercept)], yerr=[gv.sdev(intercept)], fmt='s', capsize=3, color=color)
    ax.set(title=rf'$\beta_b={beta_value(coupling):g}$, $t/a^2={time}$', xlabel=r'$1/V$', ylabel=r'$g^2_{GF}$')
handles, labels = axes.ravel()[0].get_legend_handles_labels()
fig.legend(handles, labels, ncol=3, loc='upper center', frameon=False)
fig.tight_layout(rect=(0, 0, 1, .96))
base = OUTPUT_ROOT / 'infinite_volume_all_beta_fit4'
fig.savefig(base.with_suffix('.png'), dpi=300, bbox_inches='tight')
fig.savefig(base.with_suffix('.pdf'), dpi=300, bbox_inches='tight')
plt.show(); plt.close(fig)

In [ ]:
# Infinite-volume extrapolation: beta_GF versus 1/V

fig, axes = plt.subplots(2, 4, figsize=(18, 9))

for ax, coupling in zip(axes.ravel(), COUPLINGS):
  available_times = sorted(
      bf.iv_fits[coupling]['beta'][FLOW][OBSERVABLES[0]],
      key=float,
  )
  time = min(
      available_times,
      key=lambda value: abs(float(value) - CENTRAL_WINDOW[0]),
  )

  for operator in OBSERVABLES:
      x_fit, beta_fit, beta_data = bf.infinite_volume_curve(
          coupling,
          FLOW,
          operator,
          time,
          x='beta',
      )

      x_fit = np.asarray(x_fit, dtype=float)
      beta_fit = np.asarray(beta_fit, dtype=object)
      beta_data_y = np.asarray(beta_data.y, dtype=object)

      color = OP_COLORS[operator]

      # Finite-volume beta_GF data.
      ax.errorbar(
          beta_data.x,
          gv.mean(beta_data_y),
          yerr=gv.sdev(beta_data_y),
          fmt='o',
          ms=5,
          capsize=3,
          elinewidth=1.0,
          color=color,
          label=OP_LABELS[operator],
          zorder=3,
      )

      # Infinite-volume extrapolation curve.
      ax.plot(
          x_fit,
          gv.mean(beta_fit),
          color=color,
          lw=1.6,
          zorder=2,
      )

      ax.fill_between(
          x_fit,
          gv.mean(beta_fit) - gv.sdev(beta_fit),
          gv.mean(beta_fit) + gv.sdev(beta_fit),
          color=color,
          alpha=0.16,
          linewidth=0,
          zorder=1,
      )

      # Infinite-volume intercept at 1/V = 0.
      beta_params = bf.iv_fits[coupling]['beta'][FLOW][operator][time]
      beta_infinite = bf.infinite_volume.model.evaluate(
          0.0,
          beta_params,
      )

      ax.errorbar(
          [0.0],
          [gv.mean(beta_infinite)],
          yerr=[gv.sdev(beta_infinite)],
          fmt='s',
          ms=6,
          capsize=3,
          color=color,
          markeredgecolor='black',
          markeredgewidth=0.4,
          zorder=4,
      )

  ax.set_title(
      rf'$\beta_b={beta_value(coupling):g}$, '
      rf'$t/a^2={float(time):g}$'
  )
  ax.set_xlabel(r'$1/V$')
  ax.set_ylabel(r'$\beta_{\mathrm{GF}}$')
  ax.minorticks_on()
  ax.grid(which='major', linestyle='-', alpha=0.35)
  ax.grid(which='minor', linestyle='--', alpha=0.20)

# Shared legend without duplicate entries.
handles, labels = axes.ravel()[0].get_legend_handles_labels()
unique = dict(zip(labels, handles))

fig.legend(
  unique.values(),
  unique.keys(),
  ncol=len(OBSERVABLES),
  loc='upper center',
  frameon=False,
)

fig.suptitle(
  r'Infinite-volume extrapolation of $\beta_{\mathrm{GF}}$',
  y=0.995,
)

fig.tight_layout(rect=(0, 0, 1, 0.95))

output_base = OUTPUT_ROOT / f'infinite_volume_beta_vs_invV_{FIT_ID}'
fig.savefig(
  output_base.with_suffix('.png'),
  dpi=300,
  bbox_inches='tight',
)
fig.savefig(
  output_base.with_suffix('.pdf'),
  dpi=300,
  bbox_inches='tight',
)

plt.show()
plt.close(fig)


In [ ]:
# Infinite-volume extrapolation:
# beta_GF / g_GF^4 versus 1/V

fig, axes = plt.subplots(2, 4, figsize=(18, 9))

for ax, coupling in zip(axes.ravel(), COUPLINGS):
  available_times = sorted(
      bf.iv_fits[coupling]['beta'][FLOW][OBSERVABLES[0]],
      key=float,
  )
  time = min(
      available_times,
      key=lambda value: abs(float(value) - CENTRAL_WINDOW[0]),
  )

  for operator in OBSERVABLES:
      # Retrieve the beta_GF and g_GF^2 data used in the IV fits.
      _, _, beta_data = bf.infinite_volume_curve(
          coupling,
          FLOW,
          operator,
          time,
          x='beta',
      )
      _, _, g2_data = bf.infinite_volume_curve(
          coupling,
          FLOW,
          operator,
          time,
          x='g2',
      )

      # Confirm that beta and g^2 use the same volumes in the same order.
      if beta_data.labels != g2_data.labels:
          raise RuntimeError(
              f'Volume ordering mismatch for {coupling}, '
              f'{operator}, t/a^2={time}'
          )

      inv_volume = np.asarray(beta_data.x, dtype=float)
      beta_values = np.asarray(beta_data.y, dtype=object)
      g2_values = np.asarray(g2_data.y, dtype=object)

      # Since g2_values = g_GF^2, g_GF^4 = g2_values**2.
      ratio_data = beta_values / g2_values**2

      beta_params = bf.iv_fits[coupling]['beta'][FLOW][operator][time]
      g2_params = bf.iv_fits[coupling]['g2'][FLOW][operator][time]

      x_fit = np.linspace(
          float(np.min(inv_volume)),
          float(np.max(inv_volume)),
          300,
      )

      beta_fit = np.asarray(
          [
              bf.infinite_volume.model.evaluate(x, beta_params)
              for x in x_fit
          ],
          dtype=object,
      )
      g2_fit = np.asarray(
          [
              bf.infinite_volume.model.evaluate(x, g2_params)
              for x in x_fit
          ],
          dtype=object,
      )

      ratio_fit = beta_fit / g2_fit**2
      color = OP_COLORS[operator]

      # Finite-volume beta_GF/g_GF^4 data.
      ax.errorbar(
          inv_volume,
          gv.mean(ratio_data),
          yerr=gv.sdev(ratio_data),
          fmt='o',
          ms=5,
          capsize=3,
          elinewidth=1.0,
          color=color,
          label=OP_LABELS[operator],
          zorder=3,
      )

      # Ratio of the fitted beta_GF and g_GF^2 IV curves.
      ax.plot(
          x_fit,
          gv.mean(ratio_fit),
          color=color,
          lw=1.6,
          zorder=2,
      )

      ax.fill_between(
          x_fit,
          gv.mean(ratio_fit) - gv.sdev(ratio_fit),
          gv.mean(ratio_fit) + gv.sdev(ratio_fit),
          color=color,
          alpha=0.16,
          linewidth=0,
          zorder=1,
      )

      # Infinite-volume ratio at 1/V = 0.
      beta_infinite = bf.infinite_volume.model.evaluate(
          0.0,
          beta_params,
      )
      g2_infinite = bf.infinite_volume.model.evaluate(
          0.0,
          g2_params,
      )
      ratio_infinite = beta_infinite / g2_infinite**2

      ax.errorbar(
          [0.0],
          [gv.mean(ratio_infinite)],
          yerr=[gv.sdev(ratio_infinite)],
          fmt='s',
          ms=6,
          capsize=3,
          color=color,
          markeredgecolor='black',
          markeredgewidth=0.4,
          zorder=4,
      )

  ax.set_title(
      rf'$\beta_b={beta_value(coupling):g}$, '
      rf'$t/a^2={float(time):g}$'
  )
  ax.set_xlabel(r'$1/V$')
  ax.set_ylabel(r'$\beta_{\mathrm{GF}}/g_{\mathrm{GF}}^4$')
  ax.minorticks_on()
  ax.grid(which='major', linestyle='-', alpha=0.35)
  ax.grid(which='minor', linestyle='--', alpha=0.20)

# Shared legend without duplicate entries.
handles, labels = axes.ravel()[0].get_legend_handles_labels()
unique = dict(zip(labels, handles))

fig.legend(
  unique.values(),
  unique.keys(),
  ncol=len(OBSERVABLES),
  loc='upper center',
  frameon=False,
)

fig.suptitle(
  r'Infinite-volume extrapolation of '
  r'$\beta_{\mathrm{GF}}/g_{\mathrm{GF}}^4$',
  y=0.995,
)

fig.tight_layout(rect=(0, 0, 1, 0.95))

output_base = (
  OUTPUT_ROOT
  / f'infinite_volume_beta_over_g4_vs_invV_{FIT_ID}'
)

fig.savefig(
  output_base.with_suffix('.png'),
  dpi=300,
  bbox_inches='tight',
)
fig.savefig(
  output_base.with_suffix('.pdf'),
  dpi=300,
  bbox_inches='tight',
)

plt.show()
plt.close(fig)


## Continuum scan — diagonal and kernel-correlated

Each case is serialized immediately to its own range/mode folder, matching the reference scan organization and preventing all fit covariance objects from accumulating in memory.

In [ ]:
def snapshot_case(window, mode):
    payload = {
        'window': window, 'mode': mode, 'fit_id': FIT_ID, 'order': ORDER,
        'g2s': copy.deepcopy(bf.g2s), 'betas': copy.deepcopy(bf.betas),
        'cnt_fits': copy.deepcopy(bf.cnt_fits), 'cnt_qof': copy.deepcopy(bf.continuum.quality),
        'metadata': copy.deepcopy(bf.continuum.metadata),
    }
    path = case_dir(window, mode) / 'continuum_case.gvar'
    with path.open('wb') as stream:
        gv.dump(payload, stream)

def load_case(window, mode):
    path = case_dir(window, mode) / 'continuum_case.gvar'
    with path.open('rb') as stream:
        return gv.load(stream)

mng2, mxg2, dg2 = G2_GRID
for window in WINDOWS:
    bf.cnt_xtrp(mnt=window[0], mxt=window[1], mng2=mng2, mxg2=mxg2, dg2=dg2,
                 cov_mode='diagonal', diagonal=True, correlated=False,
                 error_mode='fit', tau0=config.tau0, v=0)
    snapshot_case(window, 'diagonal')
    bf.cnt_xtrp(mnt=window[0], mxt=window[1], mng2=mng2, mxg2=mxg2, dg2=dg2,
                 cov_mode='kernel', diagonal=False, correlated=True,
                 error_mode='fit', tau0=config.tau0, v=0)
    snapshot_case(window, 'correlated')
    print('saved', window)
    gc.collect()

print(f'saved {2 * len(WINDOWS)} continuum cases')

## Plot 3 — intermediate interpolation for every flow-time range

In [ ]:
for window in WINDOWS:
     fig, ax = plt.subplots(figsize=(8, 6))

     for operator in OBSERVABLES:
         for time in flow_times(window):
             x, y, data = bf.interpolation_curve(FLOW, operator, time)
             ratio = np.asarray(y, dtype=object) / x**2
             alpha = 0.25 + .65 * (float(time) - window[0]) / max(
                 window[1] - window[0], 1e-12
             )

             ax.plot(
                 x,
                 gv.mean(ratio),
                 color=OP_COLORS[operator],
                 alpha=alpha,
                 lw=1.2,
             )
             ax.fill_between(
                 x,
                 gv.mean(ratio) - gv.sdev(ratio),
                 gv.mean(ratio) + gv.sdev(ratio),
                 color=OP_COLORS[operator],
                 alpha=.04,
             )

     xpt = np.linspace(.01, 5, 300)
     pt_curves = (
         (1, "-", "1-loop universal"),
         (2, "--", "2-loop universal"),
         (3, "-.", "3-loop gradient flow"),
     )
     for loops, ls, label in pt_curves:
         ax.plot(
             xpt,
             pt_over_g4(xpt, loops),
             color="gray",
             ls=ls,
             label=label,
         )

     # Preliminary-analysis watermark and analysis details.
     ax.text(
         0.50,
         0.52,
         r"\textbf{Preliminary}",
         transform=ax.transAxes,
         fontsize=42,
         color="gray",
         alpha=.28,
         ha="center",
         va="center",
         rotation=30,
         zorder=0,
     )
     ax.text(
         0.58,
         0.16,
         (
             FIT_WATERMARK
             + "\n"
             rf"Interpolation, $t/a^2\in[{window[0]:g},{window[1]:g}]$"
             "\n"
             r"TLN"
         ),
         transform=ax.transAxes,
         fontsize=10.5,
         color="gray",
         alpha=.52,
         ha="center",
         va="center",
         fontweight="bold",
     )

     op_handles = [
         Line2D(
             [0],
             [0],
             color=OP_COLORS[operator],
             label=OP_LABELS[operator],
         )
         for operator in OBSERVABLES
     ]
     handles, labels = ax.get_legend_handles_labels()
     ax.legend(
         op_handles + handles,
         [handle.get_label() for handle in op_handles] + labels,
         ncol=2,
         frameon=False,
     )

     ax.set(
         xlim=(0, 5),
         xlabel=r"$g^2_{GF}$",
         ylabel=r"$\beta_{GF}/g_{GF}^4$",
       #  title=rf"Interpolation, $t/a^2\in[{window[0]:g},{window[1]:g}]$",
     )

     save_figure(
         fig,
         window,
         "interpolation_beta_over_g4_fit4",
     )


In [ ]:
# Plot 3b — interpolation curves together with the fitted IV data points

  # ------------------------------------------------------------------
  # Plot controls
# ------------------------------------------------------------------
PLOT_TIME_STEP = 0.25

# Choose which operators to plot:
PLOT_OPERATORS = ('p', 'c', 's')  # all operators
# PLOT_OPERATORS = ('p',)         # Wilson only
# PLOT_OPERATORS = ('c',)         # clover only
# PLOT_OPERATORS = ('s',)         # Symanzik only


def select_flow_times(available_times, window, step):
  """Select available flow times nearest to a regular grid with spacing step."""
  available_times = sorted(available_times, key=float)

  if not available_times:
      return []

  targets = np.arange(
      window[0],
      window[1] + 0.5 * step,
      step,
  )

  selected = []
  for target in targets:
      nearest = min(
          available_times,
          key=lambda time: abs(float(time) - target),
      )

      if nearest not in selected:
          selected.append(nearest)

  return selected


for window in WINDOWS:
  fig, ax = plt.subplots(figsize=(8, 6))

  for operator in PLOT_OPERATORS:
      available_times = flow_times(window)
      times = select_flow_times(
          available_times,
          window,
          PLOT_TIME_STEP,
      )

      for time in times:
          x_grid, beta_grid, data = bf.interpolation_curve(
              FLOW,
              operator,
              time,
          )

          x_grid = np.asarray(x_grid, dtype=float)
          beta_grid = np.asarray(beta_grid, dtype=object)

          # x = g_GF^2, so g_GF^4 = x^2.
          curve_ratio = beta_grid / x_grid**2

          # Infinite-volume data points supplied to the interpolation fit.
          data_x = np.asarray(data.x, dtype=object)
          data_beta = np.asarray(data.y, dtype=object)
          data_ratio = data_beta / data_x**2

          # Later flow times are drawn more strongly.
          alpha = 0.25 + 0.65 * (
              (float(time) - window[0])
              / max(window[1] - window[0], 1e-12)
          )

          color = OP_COLORS[operator]

          # Interpolation central value.
          ax.plot(
              x_grid,
              gv.mean(curve_ratio),
              color=color,
              alpha=alpha,
              lw=1.5,
          )

          # One-sigma interpolation band.
          ax.fill_between(
              x_grid,
              gv.mean(curve_ratio) - gv.sdev(curve_ratio),
              gv.mean(curve_ratio) + gv.sdev(curve_ratio),
              color=color,
              alpha=0.08,
              linewidth=0,
          )

          # Infinite-volume data used in the fit.
          ax.errorbar(
              gv.mean(data_x),
              gv.mean(data_ratio),
              xerr=gv.sdev(data_x),
              yerr=gv.sdev(data_ratio),
              fmt="o",
              ms=4.5,
              capsize=2.5,
              elinewidth=0.9,
              color=color,
              alpha=alpha,
              markeredgecolor="black",
              markeredgewidth=0.35,
              linestyle="none",
              zorder=3,
          )

  # Perturbative reference curves.
  x_pt = np.linspace(0.01, 5.0, 300)

  pt_curves = (
      (1, "-", "1-loop universal"),
      (2, "--", "2-loop universal"),
      (3, "-.", "3-loop gradient flow"),
  )

  pt_handles = []

  for loops, linestyle, label in pt_curves:
      line, = ax.plot(
          x_pt,
          pt_over_g4(x_pt, loops),
          color="gray",
          linestyle=linestyle,
          lw=1.4,
          alpha=0.7,
          label=label,
      )
      pt_handles.append(line)

  # One legend entry for each selected operator.
  operator_handles = [
      Line2D(
          [0],
          [0],
          color=OP_COLORS[operator],
          lw=1.5,
          marker="o",
          markersize=5,
          markeredgecolor="black",
          markeredgewidth=0.35,
          label=OP_LABELS[operator],
      )
      for operator in PLOT_OPERATORS
  ]

  ax.legend(
      handles=operator_handles + pt_handles,
      ncol=2,
      frameon=False,
  )

  ax.set(
      xlim=(0, 5),
      xlabel=r"$g^2_{GF}$",
      ylabel=r"$\beta_{GF}/g_{GF}^4$",
  )

  ax.minorticks_on()
  ax.grid(which="major", linestyle="-", alpha=0.35)
  ax.grid(which="minor", linestyle="--", alpha=0.20)

  ax.text(
      0.50,
      0.52,
      r"\textbf{Preliminary}",
      transform=ax.transAxes,
      fontsize=42,
      color="gray",
      alpha=0.28,
      ha="center",
      va="center",
      rotation=30,
      zorder=0,
  )

  ax.text(
      0.58,
      0.15,
      (
          FIT_WATERMARK
          + "\n"
          + rf"Interpolation with IV data, "
          + rf"$t/a^2\in[{window[0]:g},{window[1]:g}]$"
          + "\n"
          + rf"$\Delta(t/a^2)\simeq {PLOT_TIME_STEP:g}$"
          + "\n"
          + r"TLN"
      ),
      transform=ax.transAxes,
      fontsize=10.5,
      color="gray",
      alpha=0.52,
      ha="center",
      va="center",
      fontweight="bold",
  )

  fig.tight_layout()

  save_figure(
      fig,
      window,
      "interpolation_beta_over_g4_fit4_with_data",
  )


## Plot 4 — interpolation quality for every flow-time range

In [ ]:
for window in WINDOWS:
      fig, (ax1, ax2) = plt.subplots(
          2,
          1,
          figsize=(8, 8),
          sharex=True,
      )

      for operator in OBSERVABLES:
          rows = []
          for time, qof in bf.ntrp_qof[FLOW][operator].items():
              if window[0] <= float(time) <= window[1]:
                  rows.append(
                      (
                          float(time),
                          qof['chi2'] / qof['dof']
                          if qof['dof']
                          else np.nan,
                          qof['p-value'],
                      )
                  )

          if rows:
              rows = np.asarray(sorted(rows), dtype=float)
              ax1.plot(
                  rows[:, 0],
                  rows[:, 1],
                  'o-',
                  color=OP_COLORS[operator],
                  label=OP_LABELS[operator],
              )
              ax2.plot(
                  rows[:, 0],
                  rows[:, 2],
                  'o-',
                  color=OP_COLORS[operator],
              )

      # Draw watermark above all plot elements.
      ax1.text(
          0.50,
          0.52,
          r'\textbf{Preliminary}',
          transform=ax1.transAxes,
          fontsize=42,
          color='gray',
          alpha=.28,
          ha='center',
          va='center',
          rotation=30,
          zorder=20,
          clip_on=False,
      )
      ax1.text(
          0.58,
          0.16,
          (
              FIT_WATERMARK
              + '\n'
              rf'Interpolation QoF, $t/a^2\in[{window[0]:g},{window[1]:g}]$'
              '\n'
              r'TLN'
          ),
          transform=ax1.transAxes,
          fontsize=10.5,
          color='gray',
          alpha=.52,
          ha='center',
          va='center',
          fontweight='bold',
          zorder=20,
          clip_on=False,
      )

      ax1.axhline(1, color='gray', ls='--')
      ax2.axhline(.05, color='gray', ls='--')

      ax1.set_ylabel(r'$\chi^2/{\rm dof}$')
      ax2.set_ylabel(r'$p$-value')
      ax2.set_xlabel(r'$t/a^2$')

      ax1.legend(frameon=False)

      save_figure(
          fig,
          window,
          'interpolation_quality_fit4',
      )

## Plot helper — continuum extrapolations versus $a^2/t$

In [ ]:
def plot_continuum_panels(window, mode, divide_by_g4):
      case = load_case(window, mode)
      fig, axes = plt.subplots(2, 4, figsize=(18, 9))

      tau0 = config.tau0
      correction_label = (
          'TLN'
          if config.correction in ('tln', 'tree-level-normalization')
          else 'No TLN'
      )
      mode_label = (
          'Correlated continuum'
          if mode == 'correlated'
          else 'Diagonal continuum'
      )

      for ax, target in zip(axes.ravel(), TARGET_G2):
          displayed_g2 = []

          for operator in OBSERVABLES:
              grid = np.asarray(
                  case['g2s'][FLOW][operator],
                  dtype=float,
              )

              if len(grid) == 0:
                  continue

              # Use the continuum point nearest to the requested target.
              idx = int(np.argmin(np.abs(grid - target)))
              g2 = float(grid[idx])
              displayed_g2.append(g2)

              params = case['cnt_fits'][FLOW][operator][idx]

              all_times = sorted(
                  bf.ntrp_fits[FLOW][operator],
                  key=float,
              )

              # Reproduce the exact flow-time and interpolation-domain selection
              # used by cnt_xtrp.
              fit_times = [
                  time
                  for time in all_times
                  if float(time) - tau0 > 0.0
                  and window[0] <= float(time) - tau0 <= window[1]
                  and bf.ntrp_nf[FLOW][operator][time][0]
                  <= g2
                  <= bf.ntrp_nf[FLOW][operator][time][-1]
              ]

              # Show one additional flow-time unit on either side as hollow
              # points. These points are displayed but were not used in the fit.
              outside_times = [
                  time
                  for time in all_times
                  if float(time) - tau0 > 0.0
                  and window[0] - 1.0
                  <= float(time) - tau0
                  <= window[1] + 1.0
                  and not (
                      window[0]
                      <= float(time) - tau0
                      <= window[1]
                  )
                  and bf.ntrp_nf[FLOW][operator][time][0]
                  <= g2
                  <= bf.ntrp_nf[FLOW][operator][time][-1]
              ]

              def continuum_plot_values(times):
                  nominal_times = np.asarray(
                      [float(time) - tau0 for time in times],
                      dtype=float,
                  )
                  measured_times = np.asarray(
                      [float(time) for time in times],
                      dtype=float,
                  )

                  xvalues = 1.0 / nominal_times
                  jacobian = nominal_times / measured_times

                  yvalues = np.asarray(
                      [
                          factor
                          * bf.interpolation.model.evaluate(
                              g2,
                              bf.ntrp_fits[FLOW][operator][time],
                          )
                          for factor, time in zip(jacobian, times)
                      ],
                      dtype=object,
                  )

                  norm = g2**2 if divide_byddx_g4 else 1.0
                  return xvalues, yvalues / norm

              norm = g2**2 if divide_by_g4 else 1.0

              # Filled points: data included in the continuum fit.
              if fit_times:
                  xfit_data, yfit_data = continuum_plot_values(fit_times)

                  ax.errorbar(
                      xfit_data,
                      gv.mean(yfit_data),
                      yerr=gv.sdev(yfit_data),
                      fmt='o',
                      ms=5,
                      capsize=3,
                      color=OP_COLORS[operator],
                      markerfacecolor=OP_COLORS[operator],
                      markeredgecolor=OP_COLORS[operator],
                      label=OP_LABELS[operator],
                      zorder=4,
                  )

              # Hollow points: valid neighboring points outside the fit window.
              if outside_times:
                  xoutside, youtside = continuum_plot_values(outside_times)

                  ax.errorbar(
                      xoutside,
                      gv.mean(youtside),
                      yerr=gv.sdev(youtside),
                      fmt='o',
                      ms=5,
                      capsize=3,
                      color=OP_COLORS[operator],
                      markerfacecolor='none',
                      markeredgecolor=OP_COLORS[operator],
                      alpha=.75,
                      zorder=3,
                  )

              # Draw the fitted extrapolation over the displayed a²/t range.
              displayed_times = fit_times + outside_times
              if displayed_times:
                  displayed_x = np.asarray(
                      [
                          1.0 / (float(time) - tau0)
                          for time in displayed_times
                      ],
                      dtype=float,
                  )
                  xmax = 1.05 * np.max(displayed_x)
              else:
                  xmax = 1.0 / max(window[0] - tau0, 1e-12)

              xline = np.linspace(0.0, xmax, 200)
              yline = (
                  params['beta'][0]
                  + params['slope'][0] * xline
              ) / norm

              ax.plot(
                  xline,
                  gv.mean(yline),
                  color=OP_COLORS[operator],
                  lw=1.5,
                  zorder=2,
              )
              ax.fill_between(
                  xline,
                  gv.mean(yline) - gv.sdev(yline),
                  gv.mean(yline) + gv.sdev(yline),
                  color=OP_COLORS[operator],
                  alpha=.15,
                  zorder=1,
              )

          if displayed_g2:
              panel_g2 = displayed_g2[0]
              ax.text(
                  .04,
                  .94,
                  rf'$g^2_{{GF}}={panel_g2:g}$',
                  transform=ax.transAxes,
                  ha='left',
                  va='top',
                  fontsize=13,
                  zorder=20,
              )

          ax.set_xlabel(r'$a^2/t$')

      axes.ravel()[0].legend(frameon=False)

      ylabel = (
          r'$\beta_{GF}/g_{GF}^4$'
          if divide_by_g4
          else r'$\beta_{GF}$'
      )
      fig.supylabel(ylabel)

      # Figure-level watermark and analysis information.
      fig.text(
          .50,
          .52,
          r'\textbf{Preliminary}',
          fontsize=58,
          color='gray',
          alpha=.18,
          ha='center',
          va='center',
          rotation=30,
          zorder=100,
      )
      fig.text(
          .50,
          .025,
          (
              FIT_WATERMARK
              + r'; '
              + mode_label
              + r'; '
              + rf'$t/a^2\in[{window[0]:g},{window[1]:g}]$'
              + r'; '
              + correction_label
              + '\n'
              + 'Filled points: included in fit; '
              + 'hollow points: neighboring flow times not included in fit'
          ),
          fontsize=11,
          color='gray',
          alpha=.9,
          ha='center',
          va='bottom',
          fontweight='bold',
          zorder=100,
      )

      fig.tight_layout(rect=(.025, .075, 1, 1))

      name = (
          'continuum_a2_over_t_beta_over_g4_fit4'
          if divide_by_g4
          else 'continuum_a2_over_t_beta_fit4'
      )
      save_figure(fig, window, name, mode)

## Plot 5 — raw $eta_{GF}$ continuum panels for all ranges and modes

In [ ]:
for window in WINDOWS:
    for mode in ('diagonal', 'correlated'):
        plot_continuum_panels(window, mode, divide_by_g4=False)

## Plot 6 — $eta_{GF}/g_{GF}^4$ continuum panels for all ranges and modes

In [ ]:
for window in WINDOWS:
    for mode in ('diagonal', 'correlated'):
        plot_continuum_panels(window, mode, divide_by_g4=True)

## Plot 7 — continuum fit quality for all ranges and modes

In [ ]:
for window in WINDOWS:
      for mode in ('diagonal', 'correlated'):
          case = load_case(window, mode)
          fig, (ax1, ax2) = plt.subplots(
              2,
              1,
              figsize=(8, 8),
              sharex=True,
          )

          for operator in OBSERVABLES:
              grid = np.asarray(
                  case['g2s'][FLOW][operator],
                  dtype=float,
              )
              q = case['cnt_qof'][FLOW][operator]

              chi = np.asarray(
                  [
                      row['chi2'] / row['dof']
                      if row['dof']
                      else np.nan
                      for row in q
                  ],
                  dtype=float,
              )
              pv = np.asarray(
                  [row['p-value'] for row in q],
                  dtype=float,
              )

              # Ensure that every available continuum-fit point is plotted.
              npoints = min(len(grid), len(chi), len(pv))
              grid_plot = grid[:npoints]
              chi_plot = chi[:npoints]
              pv_plot = pv[:npoints]

              ax1.plot(
                  grid_plot,
                  chi_plot,
                  '-',
                  color=OP_COLORS[operator],
                  lw=1.2,
                  alpha=.8,
              )
              ax1.scatter(
                  grid_plot,
                  chi_plot,
                  color=OP_COLORS[operator],
                  s=28,
                  zorder=3,
                  label=OP_LABELS[operator],
              )

              ax2.plot(
                  grid_plot,
                  pv_plot,
                  '-',
                  color=OP_COLORS[operator],
                  lw=1.2,
                  alpha=.8,
              )
              ax2.scatter(
                  grid_plot,
                  pv_plot,
                  color=OP_COLORS[operator],
                  s=28,
                  zorder=3,
              )

          ax1.axhline(1, color='gray', ls='--')
          ax2.axhline(.05, color='gray', ls='--')

          ax1.set_ylabel(r'$\chi^2/{\rm dof}$')
          ax2.set_ylabel(r'$p$-value')
          ax2.set_xlabel(r'$g^2_{GF}$')
          ax1.legend(frameon=False)

          correction_label = (
              'TLN'
              if config.correction in ('tln', 'tree-level-normalization')
              else 'No TLN'
          )
          mode_label = (
              'Correlated continuum QoF'
              if mode == 'correlated'
              else 'Diagonal continuum QoF'
          )

          # Watermark.
          for ax in (ax1, ax2):
              ax.text(
                  .50,
                  .52,
                  r'\textbf{Preliminary}',
                  transform=ax.transAxes,
                  fontsize=36,
                  color='gray',
                  alpha=.25,
                  ha='center',
                  va='center',
                  rotation=30,
                  zorder=0,
              )
              ax.text(
                  .58,
                  .15,
                  (
                      FIT_WATERMARK
                      + '\n'
                      + mode_label
                      + '\n'
                      + rf'$t/a^2\in[{window[0]:g},{window[1]:g}]$'
                      + '\n'
                      + correction_label
                  ),
                  transform=ax.transAxes,
                  fontsize=10.5,
                  color='gray',
                  alpha=.92,
                  ha='center',
                  va='center',
                  fontweight='bold',
              )

          save_figure(
              fig,
              window,
              'continuum_quality_fit4',
              mode,
          )


## Plot helper — final continuum curves

In [ ]:
def plot_final_curve(window, mode, divide_by_g4):
      case = load_case(window, mode)
      fig, ax = plt.subplots(figsize=(8, 6))

      correction_label = (
          'TLN'
          if config.correction in ('tln', 'tree-level-normalization')
          else 'No TLN'
      )
      mode_label = (
          'Correlated continuum'
          if mode == 'correlated'
          else 'Diagonal continuum'
      )
      quantity_label = (
          r'$\beta_{GF}/g_{GF}^4$'
          if divide_by_g4
          else r'$\beta_{GF}$'
      )

      # Continuum central curves and uncertainty bands.
      for operator in OBSERVABLES:
          x = np.asarray(
              case['g2s'][FLOW][operator],
              dtype=float,
          )
          y = np.asarray(
              case['betas'][FLOW][operator],
              dtype=object,
          )

          if divide_by_g4:
              y = y / x**2

          # Sort explicitly so the lines and bands cannot cross because of
          # an unordered coupling grid.
          order = np.argsort(x)
          x = x[order]
          y = y[order]

          ymean = gv.mean(y)
          ysdev = gv.sdev(y)

          ax.plot(
              x,
              ymean,
              color=OP_COLORS[operator],
              lw=1.8,
              label=OP_LABELS[operator],
              zorder=3,
          )
          ax.fill_between(
              x,
              ymean - ysdev,
              ymean + ysdev,
              color=OP_COLORS[operator],
              alpha=.18,
              linewidth=0,
              zorder=2,
          )

      # Perturbative comparison curves.
      xp = np.linspace(.001, 5.0, 500)
      pt_curves = (
          (1, '-', '1-loop universal'),
          (2, '--', '2-loop universal'),
          (3, '-.', '3-loop gradient flow'),
      )

      for loops, linestyle, label in pt_curves:
          perturbative_ratio = pt_over_g4(xp, loops)

          # pt_over_g4 returns beta_PT/g^4. Restore beta_PT for the
          # unscaled beta plot.
          perturbative_curve = (
              perturbative_ratio
              if divide_by_g4
              else xp**2 * perturbative_ratio
          )

          ax.plot(
              xp,
              perturbative_curve,
              color='gray',
              ls=linestyle,
              lw=1.5,
              label=label,
              zorder=1,
          )

      # Watermark drawn inside the axes and above the plotted curves.
      ax.text(
          .50,
          .52,
          r'\textbf{Preliminary}',
          transform=ax.transAxes,
          fontsize=42,
          color='gray',
          alpha=.25,
          ha='center',
          va='center',
          rotation=30,
          zorder=20,
          clip_on=True,
      )
      ax.text(
          .58,
          .15,
          (
              FIT_WATERMARK
              + '\n'
              + mode_label
              + '\n'
              + rf'$t/a^2\in[{window[0]:g},{window[1]:g}]$'
              + '\n'
              + correction_label
              + '\n'
              + quantity_label
          ),
          transform=ax.transAxes,
          fontsize=10.5,
          color='gray',
          alpha=.82,
          ha='center',
          va='center',
          fontweight='bold',
          zorder=20,
          clip_on=True,
      )

      ax.set(
          xlim=(0, 5),
          xlabel=r'$g^2_{GF}$',
          ylabel=quantity_label,
      )
      ax.legend(
          ncol=2,
          frameon=False,
      )

      name = (
          'continuum_beta_over_g4_vs_g2_fit4'
          if divide_by_g4
          else 'continuum_beta_vs_g2_fit4'
      )
      save_figure(
          fig,
          window,
          name,
          mode,
      )


## Plot 8 — final raw continuum curves for all ranges and modes

In [ ]:
for window in WINDOWS:
      for mode in ('diagonal', 'correlated'):
          plot_final_curve(
              window,
              mode,
              divide_by_g4=False,
          )


## Plot 9 — final scaled continuum curves for all ranges and modes

In [ ]:
for window in WINDOWS:
      for mode in ('diagonal', 'correlated'):
          plot_final_curve(
              window,
              mode,
              divide_by_g4=True,
          )


## Plot 10 — diagonal versus correlated continuum for every range

In [ ]:
for window in WINDOWS:
      diag = load_case(window, 'diagonal')
      corr = load_case(window, 'correlated')

      fig, ax = plt.subplots(figsize=(8, 6))

      correction_label = (
          'TLN'
          if config.correction in ('tln', 'tree-level-normalization')
          else 'No TLN'
      )

      for operator in OBSERVABLES:
          for case, linestyle, mode_label in (
              (diag, '--', 'diagonal'),
              (corr, '-', 'correlated'),
          ):
              x = np.asarray(
                  case['g2s'][FLOW][operator],
                  dtype=float,
              )
              y = (
                  np.asarray(
                      case['betas'][FLOW][operator],
                      dtype=object,
                  )
                  / x**2
              )

              # Sort the continuum grid before drawing the curve and band.
              order = np.argsort(x)
              x = x[order]
              y = y[order]

              ymean = gv.mean(y)
              ysdev = gv.sdev(y)

              # Dashed: diagonal continuum.
              # Solid: correlated continuum.
              ax.plot(
                  x,
                  ymean,
                  color=OP_COLORS[operator],
                  ls=linestyle,
                  lw=1.8,
                  label=f'{OP_LABELS[operator]} {mode_label}',
                  zorder=4 if mode_label == 'correlated' else 3,
              )

              # Uncertainty band associated with the corresponding curve.
              ax.fill_between(
                  x,
                  ymean - ysdev,
                  ymean + ysdev,
                  color=OP_COLORS[operator],
                  alpha=.18 if mode_label == 'correlated' else .07,
                  linewidth=0,
                  zorder=2 if mode_label == 'correlated' else 1,
              )

      # Perturbative beta/g^4 comparison curves.
      xp = np.linspace(.001, 5.0, 500)
      pt_curves = (
          (1, '-', '1-loop universal'),
          (2, '--', '2-loop universal'),
          (3, '-.', '3-loop gradient flow'),
      )

      for loops, linestyle, label in pt_curves:
          ax.plot(
              xp,
              pt_over_g4(xp, loops),
              color='gray',
              ls=linestyle,
              lw=1.5,
              label=label,
              zorder=1,
          )

      # Watermark and analysis information inside the plotting area.
      ax.text(
          .50,
          .52,
          r'\textbf{Preliminary}',
          transform=ax.transAxes,
          fontsize=42,
          color='gray',
          alpha=.25,
          ha='center',
          va='center',
          rotation=30,
          zorder=20,
          clip_on=True,
      )
      ax.text(
          .58,
          .15,
          (
              FIT_WATERMARK
              + '\n'
              + 'Diagonal vs correlated continuum'
              + '\n'
              + rf'$t/a^2\in[{window[0]:g},{window[1]:g}]$'
              + '\n'
              + correction_label
              + '\n'
              + r'Solid: correlated; dashed: diagonal'
          ),
          transform=ax.transAxes,
          fontsize=10.5,
          color='gray',
          alpha=.82,
          ha='center',
          va='center',
          fontweight='bold',
          zorder=20,
          clip_on=True,
      )

      ax.set(
          xlim=(0, 4.8),
          xlabel=r'$g^2_{GF}$',
          ylabel=r'$\beta_{GF}/g_{GF}^4$',
      )
      ax.legend(
          ncol=2,
          frameon=False,
      )

      save_figure(
          fig,
          window,
          'diagonal_vs_correlated_fit4',
      )


## Plot 11 — flow-time correlation matrices for every range

## Plot 12 — correlated weak-coupling extrapolation for every range

Point error bars are continuum-point uncertainties; colored bands are posterior uncertainties of the shared fitted curve and are not expected to coincide.

In [ ]:
def ratio_model(x, p):
  u = np.asarray(x) / bf.perturbative_beta_function.nrm
  correction = 1.0
  term = 1.0

  for n in range(1, ORDER + 1):
      term *= u
      correction += p[f'c{n}'][0] * term

  return pt_over_g4(x, 3) * correction


for window in WINDOWS:
  case = load_case(window, 'correlated')
  fig, ax = plt.subplots(figsize=(8, 6))

  correction_label = (
      'TLN'
      if config.correction in ('tln', 'tree-level-normalization')
      else 'No TLN'
  )

  fits = {}
  minimum_couplings = []

  for operator in OBSERVABLES:
      x = np.asarray(
          case['g2s'][FLOW][operator],
          dtype=float,
      )
      y = (
          np.asarray(
              case['betas'][FLOW][operator],
              dtype=object,
          )
          / x**2
      )

      if len(x) == 0:
          continue

      # Sort the correlated continuum results.
      order = np.argsort(x)
      x = x[order]
      y = y[order]
      minimum_couplings.append(float(x[0]))

      continuum_mean = gv.mean(y)
      continuum_sdev = gv.sdev(y)

      prior = {
          f'c{n}': [gv.gvar(0, 10)]
          for n in range(1, ORDER + 1)
      }

      # The gvars in y retain the covariance of the correlated continuum
      # calculation. lsqfit therefore uses their full covariance matrix.
      fit = lsqfit.nonlinear_fit(
          data=(x, y),
          fcn=ratio_model,
          prior=prior,
      )
      fits[operator] = fit

      # Evaluate the fitted curve and its posterior uncertainty from
      # g²=0 through the complete correlated-continuum data range.
      xp = np.linspace(0.0, float(x[-1]), 600)
      yp = np.asarray(
          ratio_model(xp, fit.p),
          dtype=object,
      )

      fit_mean = gv.mean(yp)
      fit_sdev = gv.sdev(yp)

      # Original correlated-continuum uncertainty envelope. This is shown
      # only over the region where continuum results actually exist.
      ax.plot(
          x,
          continuum_mean,
          color=OP_COLORS[operator],
          ls=':',
          lw=1.4,
          alpha=.9,
          label=(
              f'{OP_LABELS[operator]} correlated continuum '
              r'($\pm1\sigma$)'
          ),
          zorder=4,
      )
      ax.fill_between(
          x,
          continuum_mean - continuum_sdev,
          continuum_mean + continuum_sdev,
          color=OP_COLORS[operator],
          alpha=.10,
          linewidth=0,
          zorder=2,
      )

      # PT-constrained weak-coupling fit and posterior uncertainty,
      # including the extrapolated interval down to g²=0.
      ax.plot(
          xp,
          fit_mean,
          color=OP_COLORS[operator],
          ls='-',
          lw=2.0,
          label=(
              f'{OP_LABELS[operator]} correlated '
              'weak-coupling fit'
          ),
          zorder=5,
      )
      ax.fill_between(
          xp,
          fit_mean - fit_sdev,
          fit_mean + fit_sdev,
          color=OP_COLORS[operator],
          alpha=.22,
          linewidth=0,
          zorder=3,
      )

      origin = np.asarray(
          ratio_model(np.asarray([0.0]), fit.p),
          dtype=object,
      )[0]

      print(
          window,
          OP_LABELS[operator],
          'correlated',
          'origin=',
          origin,
          'Q=',
          fit.Q,
      )

  # The shaded interval has no direct correlated-continuum points.
  if minimum_couplings:
      threshold = min(minimum_couplings)
      ax.axvspan(
          0.0,
          threshold,
          color='gray',
          alpha=.08,
          label=(
              rf'extrapolated region: '
              rf'$0\leq g^2_{{GF}}<{threshold:g}$'
          ),
          zorder=0,
      )

  # Perturbative reference curves.
  xp_pt = np.linspace(0.0, 5.0, 600)
  pt_curves = (
      (1, '-', '1-loop universal'),
      (2, '--', '2-loop universal'),
      (3, '-.', '3-loop gradient flow'),
  )

  for loops, linestyle, label in pt_curves:
      ax.plot(
          xp_pt,
          pt_over_g4(xp_pt, loops),
          color='black',
          ls=linestyle,
          lw=1.4,
          alpha=.65,
          label=label,
          zorder=1,
      )

  # Watermark and analysis information inside the axes.
  ax.text(
      .50,
      .52,
      r'\textbf{Preliminary}',
      transform=ax.transAxes,
      fontsize=42,
      color='gray',
      alpha=.25,
      ha='center',
      va='center',
      rotation=30,
      zorder=20,
      clip_on=True,
  )
  ax.text(
      .58,
      .15,
      (
          FIT_WATERMARK
          + '\n'
          + 'Correlated continuum weak-coupling extrapolation'
          + '\n'
          + rf'$t/a^2\in[{window[0]:g},{window[1]:g}]$'
          + '\n'
          + correction_label
          + '\n'
          + r'Bands show posterior $\pm1\sigma$'
          + '\n'
          + r'Fixed $c_0=1$ perturbative limit'
      ),
      transform=ax.transAxes,
      fontsize=10.5,
      color='gray',
      alpha=.82,
      ha='center',
      va='center',
      fontweight='bold',
      zorder=20,
      clip_on=True,
  )

  ax.set(
      xlim=(0, 5),
      xlabel=r'$g^2_{GF}$',
      ylabel=r'$\beta_{GF}/g_{GF}^4$',
  )
  ax.legend(
      ncol=2,
      frameon=False,
  )

  save_figure(
      fig,
      window,
      'weak_coupling_to_zero_fit4',
      'correlated',
  )


## Scan manifest and validation summary

In [ ]:
rows=[]
for window in WINDOWS:
    for mode in ('diagonal','correlated'):
        case=load_case(window,mode)
        for operator in OBSERVABLES:
            q=case['cnt_qof'][FLOW][operator]
            rows.append({'tmin':window[0],'tmax':window[1],'mode':mode,'operator':operator,
                         'nfits':len(q),'mean_chi2_dof':np.mean([r['chi2']/r['dof'] for r in q if r['dof']]),
                         'mean_pvalue':np.mean([r['p-value'] for r in q])})
summary=pd.DataFrame(rows)
summary.to_csv(OUTPUT_ROOT/'fit4_scan_summary.csv',index=False)
display(summary)

expected={(window,mode) for window in WINDOWS for mode in ('diagonal','correlated')}
present={(window,mode) for window in WINDOWS for mode in ('diagonal','correlated')
         if (case_dir(window,mode)/'continuum_case.gvar').is_file()}
assert present==expected, f'missing cases: {expected-present}'
assert config.binsize==15 and config.correction=='tln'
assert config.combine=={'s': {'p':5/3,'s':-2/3}}
print('fit4 validation complete:',len(present),'continuum cases')

## Figure 11 integral matching to the perturbative regime

This adapts only the short-window matching idea of Eqs. (8)-(9) and Fig. 11 of [arXiv:2303.00704](https://arxiv.org/abs/2303.00704); the paper studied a different theory, dataset, and quantity. It is not a fit over the full NF4 continuum range. Physical inspection of the correlated NF4 continuum curves motivates the narrow trial window $g^2_{GF}\in[0.8,1.2]$, before their stronger bending develops. The $g^2=0.8$ endpoint is recomputed with the ordinary domain-restricted kernel-correlated continuum procedure. Only this narrow curve fixes the single extra coefficient $b_p$ in $\beta_4(x)=\beta_{\rm PT}^{(3)}(x)-b_p x^5$, with $x=g^2_{GF}$. Repeating the integral match after pointwise $\pm1\sigma$ shifts gives the purple limits around the blue continuation to $g^2=0$.

In [ ]:
from betafn.weak_coupling import (
    continuum_from_extended_interpolants,
    figure11_integral_match,
)

FIG11_MATCH_WINDOW = (0.8, 1.2)
FIG11_MATCH_G2 = np.linspace(*FIG11_MATCH_WINDOW, 9)
FIG11_XMAX = 3.0
fig11_results = {}

for window in WINDOWS:
    case = load_case(window, 'correlated')
    fig, axes = plt.subplots(1, 3, figsize=(17, 5.3), sharex=True, sharey=True)
    fig11_results[window] = {}

    for ax, operator in zip(axes, OBSERVABLES):
        # Recompute only the narrow matching segment with the standard
        # measured-domain restriction. This supplies the g^2=0.8 endpoint
        # without extrapolating the stored continuum curve from g^2=0.9.
        match_continuum = continuum_from_extended_interpolants(
            bf, FLOW, operator, window, FIG11_MATCH_G2, tau0=config.tau0,
            cov_mode='kernel', kernel='rbf', enforce_domains=True,
        )
        x = np.asarray(case['g2s'][FLOW][operator], dtype=float)
        y = np.asarray(case['betas'][FLOW][operator], dtype=object) / x**2
        result = figure11_integral_match(
            match_continuum['g2'], match_continuum['beta_over_g4'],
            pt_over_g4, match_window=FIG11_MATCH_WINDOW,
        )
        result['matching_continuum'] = match_continuum
        fig11_results[window][operator] = result

        order = np.argsort(x)
        x, y = x[order], y[order]
        shown = x <= FIG11_XMAX
        ax.plot(x[shown], gv.mean(y[shown]), color=OP_COLORS[operator], lw=1.8,
                label=f'{OP_LABELS[operator]} correlated continuum')
        ax.fill_between(x[shown], gv.mean(y[shown])-gv.sdev(y[shown]),
                        gv.mean(y[shown])+gv.sdev(y[shown]),
                        color=OP_COLORS[operator], alpha=.18, linewidth=0)
        mx = match_continuum['g2']
        my = match_continuum['beta_over_g4']
        ax.plot(mx, gv.mean(my), color=OP_COLORS[operator], lw=2.2)
        ax.fill_between(mx, gv.mean(my)-gv.sdev(my), gv.mean(my)+gv.sdev(my),
                        color=OP_COLORS[operator], alpha=.24, linewidth=0,
                        label=r'domain-restricted matching segment')

        xp = np.linspace(0.0, FIG11_MATCH_WINDOW[1], 500)
        central = result['ratio_curve'](xp, 'central')
        shifted_plus = result['ratio_curve'](xp, 'plus_sigma')
        shifted_minus = result['ratio_curve'](xp, 'minus_sigma')
        lower = np.minimum(shifted_plus, shifted_minus)
        upper = np.maximum(shifted_plus, shifted_minus)
        ax.fill_between(xp, lower, upper, color='purple', alpha=.24,
                        linewidth=0, label=r'matched $\pm1\sigma$ limits')
        ax.plot(xp, central, color='royalblue', lw=2.1,
                label=r'integral-matched $\beta_4$')
        ax.axvspan(*FIG11_MATCH_WINDOW, facecolor='none', edgecolor='gray',
                   hatch='///', lw=0, alpha=.45, label='matching interval')

        xp_pt = np.linspace(0.0, FIG11_XMAX, 500)
        for loops, linestyle, label in (
            (1, '-', '1-loop universal'),
            (2, '--', '2-loop universal'),
            (3, '-.', '3-loop gradient flow'),
        ):
            ax.plot(xp_pt, pt_over_g4(xp_pt, loops), color='gray',
                    ls=linestyle, lw=1.2, alpha=.75, label=label)

        coefficients = result['coefficients']
        ax.text(.50, .53, r'\textbf{Preliminary}', transform=ax.transAxes,
                fontsize=29, color='gray', alpha=.23, ha='center', va='center',
                rotation=30, zorder=20, clip_on=True)
        ax.text(.55, .13, FIT_WATERMARK + '\n'
                + rf'Fig. 11 integral match: $g^2\in[{FIG11_MATCH_WINDOW[0]:g},{FIG11_MATCH_WINDOW[1]:g}]$' + '\n'
                + rf'$t/a^2\in[{window[0]:g},{window[1]:g}]$, TLN',
                transform=ax.transAxes, fontsize=8.8, color='gray', alpha=.82,
                ha='center', va='center', fontweight='bold', zorder=20)
        ax.set(xlim=(0, FIG11_XMAX), xlabel=r'$g^2_{GF}$')
        ax.legend(fontsize=7.5, frameon=False, loc='best')
        print(window, OP_LABELS[operator], 'Fig.11 b_p:', coefficients)

    axes[0].set_ylabel(r'$\beta_{GF}/g_{GF}^4$')
    save_figure(fig, window, 'figure11_integral_matching_fit4', 'correlated')

## Diagnostic: extend intermediate interpolations first, then take the continuum limit

This is intentionally separate from both the standard continuum result and the Fig. 11 integral match. The intermediate PT-constrained curves are evaluated down to $g^2=0$ even below their measured coupling domains; at every requested $g^2$, those extrapolated finite-flow-time values are then fitted linearly in $a^2/t$ with the same kernel-correlated weighting used by the standard continuum analysis. The hollow/domain shading marks the region in which this is an interpolation-model extrapolation, so it must be treated as a model-dependence diagnostic rather than additional lattice data.

In [ ]:
from betafn.weak_coupling import continuum_from_extended_interpolants

DIRECT_G2_GRID = np.linspace(0.0, 2.0, 21)
DIRECT_DISPLAY_EPS = 1e-7
direct_weak_results = {}

for window in WINDOWS:
    correction_label = ('TLN' if config.correction in ('tln', 'tree-level-normalization')
                        else 'No TLN')

    # First show the intermediate fit curves explicitly continued to g^2=0.
    fig_intermediate, axes_intermediate = plt.subplots(1, 3, figsize=(17, 5.3),
                                                        sharex=True, sharey=True)
    xp = np.linspace(DIRECT_DISPLAY_EPS, DIRECT_G2_GRID[-1], 350)
    for ax, operator in zip(axes_intermediate, OBSERVABLES):
        times = flow_times(window)
        for index, time in enumerate(times):
            curve = np.asarray(
                bf.interpolation.model.evaluate(xp, bf.ntrp_fits[FLOW][operator][time]),
                dtype=object,
            ) / xp**2
            domain = tuple(map(float, bf.ntrp_nf[FLOW][operator][time]))
            alpha = .22 + .65 * index / max(len(times)-1, 1)
            ax.plot(xp, gv.mean(curve), color=OP_COLORS[operator], alpha=alpha, lw=1.1)
            ax.axvline(domain[0], color=OP_COLORS[operator], alpha=.10, lw=.7)
        for loops, linestyle, label in (
            (1, '-', '1-loop universal'), (2, '--', '2-loop universal'),
            (3, '-.', '3-loop gradient flow'),
        ):
            ax.plot(xp, pt_over_g4(xp, loops), color='gray', ls=linestyle,
                    lw=1.2, alpha=.75, label=label)
        ax.text(.50, .53, r'\textbf{Preliminary}', transform=ax.transAxes,
                fontsize=29, color='gray', alpha=.23, ha='center', va='center',
                rotation=30, zorder=20, clip_on=True)
        ax.text(.55, .13, FIT_WATERMARK + '\nExtended intermediate interpolation' + '\n'
                + rf'$t/a^2\in[{window[0]:g},{window[1]:g}]$, {correction_label}',
                transform=ax.transAxes, fontsize=8.8, color='gray', alpha=.82,
                ha='center', va='center', fontweight='bold', zorder=20)
        ax.set(xlim=(0, DIRECT_G2_GRID[-1]), xlabel=r'$g^2_{GF}$')
        ax.legend(fontsize=8, frameon=False)
    axes_intermediate[0].set_ylabel(r'$\beta_{GF}/g_{GF}^4$')
    save_figure(fig_intermediate, window,
                'extended_intermediate_to_zero_fit4', 'correlated_diagnostic')

    # Then take a separate correlated continuum limit of those extended curves.
    fig_continuum, axes_continuum = plt.subplots(1, 3, figsize=(17, 5.3),
                                                    sharex=True, sharey=True)
    direct_weak_results[window] = {}
    for ax, operator in zip(axes_continuum, OBSERVABLES):
        result = continuum_from_extended_interpolants(
            bf, FLOW, operator, window, DIRECT_G2_GRID, tau0=config.tau0,
            cov_mode='kernel', kernel='rbf',
        )
        direct_weak_results[window][operator] = result
        x = result['g2']
        y = result['beta_over_g4']
        fully_supported_from = max(domain[0] for domain in result['measured_domains'].values())
        ax.axvspan(0, min(fully_supported_from, x[-1]), color='gray', alpha=.10,
                   label='intermediate-fit extrapolation')
        ax.plot(x, gv.mean(y), color=OP_COLORS[operator], lw=2.0,
                label=f'{OP_LABELS[operator]} extended continuum')
        ax.fill_between(x, gv.mean(y)-gv.sdev(y), gv.mean(y)+gv.sdev(y),
                        color=OP_COLORS[operator], alpha=.22, linewidth=0,
                        label=r'continuum $\pm1\sigma$')
        xp_pt = np.linspace(0.0, DIRECT_G2_GRID[-1], 400)
        for loops, linestyle, label in (
            (1, '-', '1-loop universal'), (2, '--', '2-loop universal'),
            (3, '-.', '3-loop gradient flow'),
        ):
            ax.plot(xp_pt, pt_over_g4(xp_pt, loops), color='gray',
                    ls=linestyle, lw=1.2, alpha=.75, label=label)
        ax.text(.50, .53, r'\textbf{Preliminary}', transform=ax.transAxes,
                fontsize=29, color='gray', alpha=.23, ha='center', va='center',
                rotation=30, zorder=20, clip_on=True)
        ax.text(.55, .13, FIT_WATERMARK + '\nExtended-interpolation continuum diagnostic' + '\n'
                + rf'$t/a^2\in[{window[0]:g},{window[1]:g}]$, {correction_label}',
                transform=ax.transAxes, fontsize=8.8, color='gray', alpha=.82,
                ha='center', va='center', fontweight='bold', zorder=20)
        ax.set(xlim=(0, DIRECT_G2_GRID[-1]), xlabel=r'$g^2_{GF}$')
        ax.legend(fontsize=8, frameon=False)
        print(window, OP_LABELS[operator], 'direct weak continuum QoF:',
              pd.DataFrame(result['quality']).to_string(index=False))
    axes_continuum[0].set_ylabel(r'$\beta_{GF}/g_{GF}^4$')
    save_figure(fig_continuum, window,
                'extended_interpolation_continuum_to_zero_fit4',
                'correlated_diagnostic')